# Sparkfun Crawled EAGLE / KiCad Zip link

In [24]:
import csv
import time
import requests
from urllib.parse import urlparse, parse_qs, urlunparse, urlencode, urljoin
from lxml import html
from requests.adapters import HTTPAdapter, Retry

# ---------- HTTP session with retries ----------
def _session_with_retries(total=3, backoff=0.5, timeout=20):
    s = requests.Session()
    retries = Retry(
        total=total,
        backoff_factor=backoff,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(['HEAD','GET','OPTIONS'])
    )
    s.mount('http://', HTTPAdapter(max_retries=retries))
    s.mount('https://', HTTPAdapter(max_retries=retries))
    s.headers.update({
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/124.0.0.0 Safari/537.36")
    })
    s.request_timeout = timeout
    return s

# ---------- Utilities ----------
def _normalize_to_page1(url: str) -> str:
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    qs.pop('p', None)
    return urlunparse(parsed._replace(query=urlencode({k: v[0] for k, v in qs.items()})))

def _is_empty_page(tree) -> bool:
    """
    Detect Magento 'no products' message:
    <div class="message info empty"><div>We can't find products matching the selection.</div></div>
    """
    return bool(tree.xpath("//div[contains(@class,'message') and contains(@class,'info') and contains(@class,'empty')]"))

def _iter_pages_until_empty(category_url: str, sess: requests.Session, max_pages: int = 200):
    """
    Yield page URLs from ?p=1 increasing until an empty page is encountered,
    or until max_pages is reached (safety cap).
    """
    base_url = _normalize_to_page1(category_url)
    parsed = urlparse(base_url)

    for i in range(1, max_pages + 1):
        qs = parse_qs(parsed.query)
        qs['p'] = [str(i)]
        page_url = urlunparse(parsed._replace(query=urlencode({k: v[0] for k, v in qs.items()})))

        r = sess.get(page_url, timeout=sess.request_timeout)
        r.raise_for_status()
        tree = html.fromstring(r.content)

        if _is_empty_page(tree):
            break  # stop when Magento shows the empty message

        yield page_url

def _get_product_links_from_page(page_url: str, sess: requests.Session) -> list[str]:
    r = sess.get(page_url, timeout=sess.request_timeout)
    r.raise_for_status()
    tree = html.fromstring(r.content)

    anchors = tree.xpath('//*[@id="amasty-shopby-product-list"]/div[2]/ol//a[@href]')
    seen, out = set(), []
    for a in anchors:
        href = a.get('href')
        if href:
            absolute = urljoin(page_url, href)
            if absolute not in seen:
                seen.add(absolute)
                out.append(absolute)
    return out

def _find_kicad_eagle_links(product_url: str, sess: requests.Session):
    """
    Return (kicad_url, eagle_url)
    - kicad_url: .zip whose anchor text contains 'kicad'
    - eagle_url: .zip whose anchor text contains 'eagle', else 3rd .zip found if exists
    """
    r = sess.get(product_url, timeout=sess.request_timeout)
    r.raise_for_status()
    tree = html.fromstring(r.content)

    zip_links_in_order = []
    kicad_url = ""
    eagle_url = ""

    for a in tree.xpath('//a[@href]'):
        href = (a.get('href') or '').strip()
        if not href.lower().endswith('.zip'):
            continue
        abs_url = urljoin(product_url, href)
        zip_links_in_order.append(abs_url)

        text = (a.text_content() or '').strip().lower()
        if not kicad_url and "kicad" in text:
            kicad_url = abs_url
        if not eagle_url and "eagle" in text:
            eagle_url = abs_url

    # Deduplicate while preserving order
    seen = set()
    zip_links_in_order = [u for u in zip_links_in_order if not (u in seen or seen.add(u))]

    # Eagle fallback
    if not eagle_url and len(zip_links_in_order) >= 3:
        eagle_url = zip_links_in_order[2]

    return kicad_url, eagle_url

# ---------- Main Function ----------
def category_to_csv(category_url: str, csv_path: str, delay_sec: float = 0.0):
    """
    Crawl category URL and save CSV with columns:
    page_url, product_url, kicad_url, eagle_url
    Iterates pages sequentially (?p=1,2,3,...) until the 'empty' message appears.
    """
    sess = _session_with_retries()

    page_count = 0
    product_count = 0

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["page_url", "product_url", "kicad_url", "eagle_url"])

        for page_url in _iter_pages_until_empty(category_url, sess):
            page_count += 1
            try:
                product_links = _get_product_links_from_page(page_url, sess)
                product_count += len(product_links)
                print(f"[Page {page_count}] {page_url} -> {len(product_links)} products")
            except Exception as e:
                print(f"[Page error] {page_url} -> {e}")
                continue

            if delay_sec:
                time.sleep(delay_sec)

            for product_url in product_links:
                try:
                    kicad_url, eagle_url = _find_kicad_eagle_links(product_url, sess)
                except Exception as e:
                    print(f"[Product error] {product_url} -> {e}")
                    kicad_url, eagle_url = "", ""

                writer.writerow([page_url, product_url, kicad_url, eagle_url])

                if delay_sec:
                    time.sleep(delay_sec)

    print(f"Done. Pages crawled: {page_count}, products seen: {product_count}")


def csv_path_from_url(url, base_folder):
    """Convert category URL to CSV path inside base_folder."""
    path_name = os.path.basename(urlparse(url).path)  # e.g., "components.html"
    if path_name.endswith(".html"):
        path_name = path_name[:-5]  # remove ".html"
    file_name = f"sparkfun_{path_name}_zips.csv"
    return os.path.join(base_folder, file_name)

# Example

In [ ]:
sparkfun_category_url =['https://www.sparkfun.com/audio.html',
                        'https://www.sparkfun.com/components.html',
                        'https://www.sparkfun.com/data-logging-and-memory.html',
                        'https://www.sparkfun.com/development-boards.html',
                        'https://www.sparkfun.com/displays.html',
                        'https://www.sparkfun.com/e-textiles-crafting.html',
                        'https://www.sparkfun.com/gnss.html',
                        'https://www.sparkfun.com/iot-wireless.html',
                        'https://www.sparkfun.com/kits.html',
                        'https://www.sparkfun.com/robotics.html',
                        'https://www.sparkfun.com/sensors.html',
                        'https://www.sparkfun.com/tools.html',
                        'https://www.sparkfun.com/special-categories.html']

In [27]:
base_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun URL"
cat_url = 'https://www.sparkfun.com/special-categories.html'
category_to_csv(
    cat_url,
    csv_path_from_url(cat_url, base_folder),
    delay_sec=0.2        # polite throttling
)

[Page 1] https://www.sparkfun.com/special-categories.html?p=1 -> 25 products
[Page 2] https://www.sparkfun.com/special-categories.html?p=2 -> 25 products
[Page 3] https://www.sparkfun.com/special-categories.html?p=3 -> 25 products
[Page 4] https://www.sparkfun.com/special-categories.html?p=4 -> 25 products
[Page 5] https://www.sparkfun.com/special-categories.html?p=5 -> 25 products
[Page 6] https://www.sparkfun.com/special-categories.html?p=6 -> 25 products
[Page 7] https://www.sparkfun.com/special-categories.html?p=7 -> 25 products
[Page 8] https://www.sparkfun.com/special-categories.html?p=8 -> 25 products
[Page 9] https://www.sparkfun.com/special-categories.html?p=9 -> 25 products
[Page 10] https://www.sparkfun.com/special-categories.html?p=10 -> 25 products
[Page 11] https://www.sparkfun.com/special-categories.html?p=11 -> 25 products
[Page 12] https://www.sparkfun.com/special-categories.html?p=12 -> 25 products
[Page 13] https://www.sparkfun.com/special-categories.html?p=13 -> 25 

In [22]:
base_folder = "/Users/taitinglu/Desktop/IMG2SCH/Sparkfun URL"

for cat_url in sparkfun_category_url:
    category_to_csv(
        cat_url,
        csv_path_from_url(cat_url, base_folder),
        delay_sec=0.2        # polite throttling
    )

[Page 1] https://www.sparkfun.com/data-logging-and-memory.html?p=1 -> 25 products
[Page 2] https://www.sparkfun.com/data-logging-and-memory.html?p=2 -> 22 products
Done. Pages crawled: 2, products seen: 47
[Page 1] https://www.sparkfun.com/development-boards.html?p=1 -> 25 products
[Page 2] https://www.sparkfun.com/development-boards.html?p=2 -> 25 products
[Page 3] https://www.sparkfun.com/development-boards.html?p=3 -> 25 products
[Page 4] https://www.sparkfun.com/development-boards.html?p=4 -> 25 products
[Page 5] https://www.sparkfun.com/development-boards.html?p=5 -> 25 products
[Page 6] https://www.sparkfun.com/development-boards.html?p=6 -> 25 products
[Page 7] https://www.sparkfun.com/development-boards.html?p=7 -> 25 products
[Page 8] https://www.sparkfun.com/development-boards.html?p=8 -> 25 products
[Page 9] https://www.sparkfun.com/development-boards.html?p=9 -> 25 products
[Page 10] https://www.sparkfun.com/development-boards.html?p=10 -> 25 products
[Page 11] https://www.

# Download Zip 

In [32]:
import os
import csv
import pathlib
import requests
from urllib.parse import urlparse

def download_from_csv(csv_path, dest_folder, which="eagle", overwrite=False, timeout=30):
    """
    Download ZIPs listed in a CSV produced by your scraper.
    Saves each file into its own subfolder named after the ZIP filename (without extension).

    Args:
        csv_path (str): Path to CSV with columns: page_url, product_url, kicad_url, eagle_url
        dest_folder (str): Directory to save files
        which (str): 'eagle', 'kicad', or 'both' (case-insensitive)
        overwrite (bool): Overwrite existing files if True
        timeout (int): Request timeout (seconds)

    Returns:
        list[str]: Paths to saved ZIP files
    """
    which = which.lower()
    assert which in {"eagle", "kicad", "both"}, "which must be 'eagle', 'kicad', or 'both'"

    os.makedirs(dest_folder, exist_ok=True)
    saved = []

    headers = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/124.0.0.0 Safari/537.36")
    }

    # ---- collect links from CSV (dedup + keep order) ----
    links = []
    seen = set()
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        # Expect columns: page_url, product_url, kicad_url, eagle_url
        for row in reader:
            if which in {"kicad", "both"}:
                url = (row.get("kicad_url") or "").strip()
                if url and url not in seen:
                    seen.add(url)
                    links.append(url)
            if which in {"eagle", "both"}:
                url = (row.get("eagle_url") or "").strip()
                if url and url not in seen:
                    seen.add(url)
                    links.append(url)

    total = len(links)
    for idx, url in enumerate(links, start=1):
        try:
            base_name = pathlib.Path(urlparse(url).path).name or "file.zip"
            name_stem, name_ext = os.path.splitext(base_name)

            # Create a folder for this zip file
            zip_folder = os.path.join(dest_folder, name_stem)
            os.makedirs(zip_folder, exist_ok=True)
            dest_path = os.path.join(zip_folder, base_name)

            # If overwrite=False and file exists, append '##<n>' before extension
            if os.path.exists(dest_path) and not overwrite:
                counter = 1
                while os.path.exists(dest_path):
                    new_name = f"{name_stem}##{counter}{name_ext}"
                    dest_path = os.path.join(zip_folder, new_name)
                    counter += 1

            print(f"[{idx}/{total}] Downloading: {os.path.basename(dest_path)} -> {zip_folder}")

            with requests.get(url, headers=headers, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                tmp_path = dest_path + ".part"
                with open(tmp_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 64):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp_path, dest_path)

            saved.append(dest_path)

        except Exception as e:
            print(f"[{idx}/{total}] Skipping failed download {url}: {e}")
            continue

    return saved


# Example

In [34]:
# --- Examples ---
download_from_csv("/Users/taitinglu/Desktop/IMG2SCH/Sparkfun URL/sparkfun_special-categories_zips.csv", 
"/Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_special-categories", 
which="eagle")
# download_from_csv("sparkfun_audio_links.csv", "./zips_kicad", which="kicad")
# download_from_csv("sparkfun_audio_links.csv", "./zips_both", which="both")

[1/568] Downloading: SparkFun_AST-CAN485_1.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_special-categories/SparkFun_AST-CAN485_1
[2/568] Downloading: SparkFun_Thing_Plus_ESP32_WROOM_C_eagle_files2.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_special-categories/SparkFun_Thing_Plus_ESP32_WROOM_C_eagle_files2
[3/568] Downloading: MicroMod_Asset_Tracker.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_special-categories/MicroMod_Asset_Tracker
[4/568] Downloading: 17731-QwiicSoilMoistureSensor-EagleFiles.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_special-categories/17731-QwiicSoilMoistureSensor-EagleFiles
[5/568] Downloading: 19177_SparkFun_IoT_RedBoard-ESP32-EagleFiles.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_special-categories/19177_SparkFun_IoT_RedBoard-ESP32-EagleFiles
[6/568] Downloading: SparkFun_ENS160_BME280_Breakout_v11.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_special-categories/SparkFun_ENS16

KeyboardInterrupt: 

In [36]:
sparkfun_category_url =['https://www.sparkfun.com/audio.html',
                        'https://www.sparkfun.com/components.html',
                        'https://www.sparkfun.com/data-logging-and-memory.html',
                        'https://www.sparkfun.com/development-boards.html',
                        'https://www.sparkfun.com/displays.html',
                        'https://www.sparkfun.com/e-textiles-crafting.html',
                        'https://www.sparkfun.com/gnss.html',
                        'https://www.sparkfun.com/iot-wireless.html',
                        'https://www.sparkfun.com/kits.html',
                        'https://www.sparkfun.com/robotics.html',
                        'https://www.sparkfun.com/sensors.html',
                        'https://www.sparkfun.com/tools.html',
                        'https://www.sparkfun.com/special-categories.html']


for cat_url in sparkfun_category_url:
    path_name = os.path.basename(urlparse(cat_url).path)[:-5]  # remove ".html"
    csv_path = csv_path_from_url(cat_url, base_folder)
    print(f"Downloading from {csv_path}")
    download_from_csv(csv_path, 
        f"/Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_{path_name}", 
        which="eagle")

[1/28] Downloading: SparkFun_Tsunami_Qwiic_Eagle_Files_V21.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_audio/SparkFun_Tsunami_Qwiic_Eagle_Files_V21
[2/28] Downloading: mp3-trigger-v24.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_audio/mp3-trigger-v24
[3/28] Downloading: STA540%20Audio%20Amp%20v11.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_audio/STA540%20Audio%20Amp%20v11
[4/28] Downloading: Stereo_Audio_Codec_Breakout_WM8960_v10.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_audio/Stereo_Audio_Codec_Breakout_WM8960_v10
[5/28] Downloading: SparkFun_Qwiic_Buzzer_V10.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_audio/SparkFun_Qwiic_Buzzer_V10
[6/28] Downloading: SparkFun_Noisy_Cricket_1.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_audio/SparkFun_Noisy_Cricket_1
[7/28] Downloading: SparkFun_Qwiic_Speaker_Amp-TPA2016D2_v10.zip -> /Users/taitinglu/Desktop/IMG2SCH/eagle zip/sparkfun_audio/SparkFun_Qwiic_Speak